# HidroVision · Fine-tuning na régua própria

Parte do `hidrovision_v05.pt` e especializa o modelo nas 382 imagens da régua fabricada pela equipe (lote 09/08 com 160 + lote NemData com 222, em dois cenários).

**Antes de rodar**
1. No Roboflow, em *Dataset*, filtre pela tag `regua_propria`, selecione as 382 imagens e use **Export** no formato YOLOv8.
2. Suba o zip exportado e o `hidrovision_v05.pt` como datasets no Kaggle (Add Input) e ajuste os caminhos abaixo.
3. Acelerador **GPU T4**.

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, glob, yaml, shutil, random, zipfile
from pathlib import Path
from ultralytics import YOLO

ORIGEM = "/kaggle/input/regua-propria"
PESOS_V05 = "/kaggle/input/hidrovision-v05/hidrovision_v05.pt"
SAIDA = Path("/kaggle/working")
RAIZ = SAIDA / "regua_propria"
SEMENTE = 42

## 1. Localizar imagens e rótulos do export

In [ ]:
bruto = SAIDA / "bruto"
shutil.rmtree(bruto, ignore_errors = True)
bruto.mkdir()

for z in glob.glob(f"{ORIGEM}/**/*.zip", recursive = True):
    zipfile.ZipFile(z).extractall(bruto)
if not any(bruto.iterdir()):
    shutil.copytree(ORIGEM, bruto, dirs_exist_ok = True)

ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
imagens = [p for p in bruto.rglob("*") if p.suffix.lower() in ext]
rotulos = {p.stem: p for p in bruto.rglob("*.txt") if p.parent.name == "labels"}
yamls = list(bruto.rglob("data.yaml"))

print("imagens:", len(imagens))
print("rótulos:", len(rotulos))
print("data.yaml:", yamls)

## 2. Conferências e remapeamento de classes

O V05 define a ordem das 13 classes. O export pode vir com outra ordem ou com classes extras (como o `5`), então cada rótulo é remapeado **pelo nome** para o índice do V05 e classes que o V05 não conhece são descartadas.

In [ ]:
v05 = YOLO(PESOS_V05)
nomes_v05 = [v05.names[i] for i in sorted(v05.names)]

with open(yamls[0]) as f:
    cfg = yaml.safe_load(f)
nomes_exp = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]

mapa = {i: nomes_v05.index(n) for i, n in enumerate(nomes_exp) if n in nomes_v05}
fora = [n for n in nomes_exp if n not in nomes_v05]

print("V05:   ", nomes_v05)
print("export:", nomes_exp)
print("descartadas:", fora)
assert len(mapa) == 13, "export não contém as 13 classes do V05"

def converter(linha):
    p = linha.split()
    cls = int(p[0])
    if cls not in mapa:
        return None
    if len(p) > 5:
        xs = [float(v) for v in p[1::2]]
        ys = [float(v) for v in p[2::2]]
        x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
        return f"{mapa[cls]} {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}"
    return f"{mapa[cls]} {' '.join(p[1:5])}"

pares, vazias = [], []
for img in imagens:
    lab = rotulos.get(img.stem)
    linhas = []
    if lab:
        linhas = [converter(l) for l in lab.read_text().splitlines() if l.strip()]
        linhas = [l for l in linhas if l]
    (pares if linhas else vazias).append((img, linhas))

import hashlib
vistos, unicos, repetidas = set(), [], 0
for img, linhas in pares:
    h = hashlib.md5(img.read_bytes()).hexdigest()
    if h in vistos:
        repetidas += 1
        continue
    vistos.add(h)
    unicos.append((img, linhas))
pares = unicos

print(f"anotadas: {len(pares)}   sem anotação: {len(vazias)}   duplicadas removidas: {repetidas}")
assert len(pares) > 0.9 * len(imagens), "mais de 10% das imagens sem anotação: confira no Roboflow antes de treinar"

## 3. Divisão 70 / 20 / 10

Sorteio com semente fixa, então a divisão é reproduzível. As imagens sem anotação ficam de fora.

In [ ]:
random.seed(SEMENTE)
random.shuffle(pares)
n = len(pares)
cortes = {"train": pares[:int(0.7 * n)], "valid": pares[int(0.7 * n):int(0.9 * n)], "test": pares[int(0.9 * n):]}

shutil.rmtree(RAIZ, ignore_errors = True)
for split, itens in cortes.items():
    (RAIZ / split / "images").mkdir(parents = True)
    (RAIZ / split / "labels").mkdir(parents = True)
    for img, linhas in itens:
        shutil.copy(img, RAIZ / split / "images" / img.name)
        (RAIZ / split / "labels" / f"{img.stem}.txt").write_text("\n".join(linhas) + "\n")
    print(f"{split:6s} {len(itens):4d}")

DATA = RAIZ / "data.yaml"
yaml.safe_dump({"path": str(RAIZ), "train": "train/images", "val": "valid/images", "test": "test/images", "nc": 13, "names": nomes_v05}, open(DATA, "w"), allow_unicode = True)
print(DATA.read_text())

## 3. Referência: V05 original na régua própria

Esse é o número a bater. Se o V05 já estiver alto aqui, o ganho do fine-tuning vai aparecer principalmente no recall dos números.

In [ ]:
r_v05 = v05.val(data = str(DATA), split = "test", imgsz = 640, batch = 16, device = 0, plots = False, verbose = False, project = str(SAIDA / "val"), name = "v05")
print(f"V05  mAP50 {r_v05.box.map50:.3f}  mAP50-95 {r_v05.box.map:.3f}  P {r_v05.box.mp:.3f}  R {r_v05.box.mr:.3f}")

## 4. Fine-tuning

- `optimizer = "AdamW"` com `lr0 = 0.001`: taxa 10× menor que a do treino original. O otimizador precisa ser fixo, porque com `optimizer = auto` o Ultralytics ignora o `lr0`.
- `freeze = 10`: congela o backbone (extrator de formas) e treina só o pescoço e a cabeça.
- `patience = 20`: para sozinho se a validação não melhorar.
- `hsv_v = 0.3`: variação de brilho de ±30%, a mesma receita usada no Roboflow para o V05.
- `fliplr = 0`: régua espelhada não existe e números espelhados confundem o modelo.

In [ ]:
modelo = YOLO(PESOS_V05)
modelo.train(
    data = str(DATA),
    epochs = 60,
    imgsz = 640,
    batch = 16,
    optimizer = "AdamW",
    lr0 = 0.001,
    lrf = 0.01,
    warmup_epochs = 3,
    patience = 20,
    freeze = 10,
    hsv_v = 0.3,
    fliplr = 0.0,
    mosaic = 1.0,
    close_mosaic = 10,
    device = 0,
    project = str(SAIDA / "runs"),
    name = "ft_regua",
    exist_ok = True,
    plots = True
)

## 5. Comparação no conjunto de teste

In [ ]:
BEST = SAIDA / "runs" / "ft_regua" / "weights" / "best.pt"
ft = YOLO(str(BEST))
r_ft = ft.val(data = str(DATA), split = "test", imgsz = 640, batch = 16, device = 0, plots = True, verbose = False, project = str(SAIDA / "val"), name = "ft")

print(f"{'':10s}{'mAP50':>8s}{'mAP50-95':>10s}{'P':>8s}{'R':>8s}")
for nome, r in [("V05", r_v05), ("ajustado", r_ft)]:
    print(f"{nome:10s}{r.box.map50:8.3f}{r.box.map:10.3f}{r.box.mp:8.3f}{r.box.mr:8.3f}")

In [ ]:
import pandas as pd

def por_classe(r):
    return {r.names[int(i)]: (r.box.ap50[k], r.box.r[k]) for k, i in enumerate(r.box.ap_class_index)}

a, b = por_classe(r_v05), por_classe(r_ft)
linhas = []
for cls in nomes_v05:
    if cls in a or cls in b:
        ap_a, rc_a = a.get(cls, (float("nan"), float("nan")))
        ap_b, rc_b = b.get(cls, (float("nan"), float("nan")))
        linhas.append({"classe": cls, "mAP50 V05": ap_a, "mAP50 ajustado": ap_b, "recall V05": rc_a, "recall ajustado": rc_b, "ganho recall": rc_b - rc_a})

tabela = pd.DataFrame(linhas).round(3)
tabela.to_csv(SAIDA / "comparacao_por_classe.csv", index = False)
tabela

## 6. Inspeção visual em imagens de teste

In [ ]:
from IPython.display import Image, display

testes = sorted(glob.glob(str(RAIZ / "test" / "images" / "*")))[:6]
for m, nome in [(v05, "v05"), (ft, "ft")]:
    m.predict(testes, imgsz = 640, conf = 0.25, save = True, project = str(SAIDA / "pred"), name = nome, exist_ok = True, verbose = False)

for img in testes[:3]:
    nome = Path(img).name
    print(nome)
    display(Image(filename = str(SAIDA / "pred" / "v05" / nome), width = 420))
    display(Image(filename = str(SAIDA / "pred" / "ft" / nome), width = 420))

## 7. Exportar

In [ ]:
shutil.copy(BEST, SAIDA / "hidrovision_v06_regua.pt")
ft.export(format = "onnx", imgsz = 640, simplify = True)
shutil.copy(BEST.with_suffix(".onnx"), SAIDA / "hidrovision_v06_regua.onnx")
print(sorted(os.listdir(SAIDA)))